### Procesamiento de Lenguaje Natural I
# **Desafío 1**



### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [1]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [2]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [3]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [4]:
tfidfvect = TfidfVectorizer()

Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [5]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Es muy útil tener el diccionario opuesto que va de índices a términos

In [6]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [7]:
y_train = newsgroups_train.target


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [8]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [9]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [10]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


## Parte 1 — Similaridad de 5 documentos al azar


Tomamos 5 documentos al azar del set de entrenamiento (con semilla fija para reproducibilidad), calculamos su similaridad coseno contra todo el corpus de train y nos quedamos con los 5 más parecidos a cada uno (excluyendo el propio documento). Para cada par documento-vecino mostramos la clase asignada, así podemos juzgar si la similaridad capturada por TF-IDF se condice con la categoría temática del newsgroup.


In [11]:
rng = np.random.default_rng(seed=42)
random_idxs = rng.choice(X_train.shape[0], size=5, replace=False)
random_idxs


array([8754, 4965, 7404, 1009, 4899])

In [12]:
def top_n_similares(idx, X, n=5):
    sims = cosine_similarity(X[idx], X)[0]
    orden = np.argsort(sims)[::-1]
    # excluimos el propio documento (siempre primero, similaridad 1.0)
    vecinos = orden[orden != idx][:n]
    return vecinos, sims[vecinos]


In [13]:
for idx in random_idxs:
    clase_original = newsgroups_train.target_names[y_train[idx]]
    vecinos, sims = top_n_similares(idx, X_train, n=5)
    print('=' * 80)
    print(f'DOC #{idx} | clase: {clase_original}')
    print('-' * 80)
    texto = newsgroups_train.data[idx].strip().replace('\n', ' ')
    print(f'Texto (primeros 300 chars): {texto[:300]}...')
    print('-' * 80)
    print('Top-5 vecinos:')
    for v, s in zip(vecinos, sims):
        clase_v = newsgroups_train.target_names[y_train[v]]
        match = '✔' if clase_v == clase_original else '✘'
        print(f'  {match} doc #{v:5d} | sim={s:.3f} | clase: {clase_v}')
    print()


DOC #8754 | clase: talk.religion.misc
--------------------------------------------------------------------------------
Texto (primeros 300 chars): /(hudson) /If someone inflicts pain on themselves, whether they enjoy it or not, they /are hurting themselves.  They may be permanently damaging their body.  That is true.  It is also none of your business.    Some people may also reason that by reading the bible and being a Xtian you are permanentl...
--------------------------------------------------------------------------------
Top-5 vecinos:
  ✔ doc # 6552 | sim=0.490 | clase: talk.religion.misc
  ✔ doc #10613 | sim=0.481 | clase: talk.religion.misc
  ✔ doc # 3616 | sim=0.465 | clase: talk.religion.misc
  ✘ doc # 8726 | sim=0.460 | clase: talk.politics.mideast
  ✔ doc # 3902 | sim=0.459 | clase: talk.religion.misc

DOC #4965 | clase: comp.sys.mac.hardware
--------------------------------------------------------------------------------
Texto (primeros 300 chars): No.  Plug the printer in

### Interpretación — Parte 1

Acierto agregado: **13 / 25 vecinos coinciden con la clase del documento original** (52 %). Pero el desglose por documento es lo interesante:

- **DOC #8754 (`talk.religion.misc`)** — 4/5 aciertos. El único vecino "equivocado" es `talk.politics.mideast`, y revisando el texto tiene sentido: el doc mezcla biblia con política, y ambas categorías comparten vocabulario ideológico.
- **DOC #4965 (`comp.sys.mac.hardware`)** — 2/5 aciertos. El texto es **una sola línea** ("Plug the printer in the printer port, and the modem in the modem port"), así que el vector tiene muy poca información y los vecinos se van a clases hermanas (`comp.sys.ibm.pc.hardware`, `comp.graphics`). Confirma que con docs cortos el TF-IDF se degrada.
- **DOC #7404 (`comp.os.ms-windows.misc`)** — **0/5 aciertos**. Caso patológico: el texto vuelve a ser corto y los vecinos caen en `comp.windows.x` (que comparte vocabulario casi total con `comp.os.ms-windows.misc`). Acá la confusión es **inevitable a nivel léxico** — son dos newsgroups que hablan de lo mismo con etiquetas distintas.
- **DOC #1009 (`talk.politics.guns`)** — 4/5 aciertos. El único error cae en `alt.atheism`, otra vez por vocabulario ideológico compartido.
- **DOC #4899 (`sci.crypt`)** — 3/5 aciertos. Curiosidad: el documento ruidoso `doc #8726` (`talk.politics.mideast`) aparece como vecino de **dos** docs distintos (este y el #8754). Probablemente es un "hub" con vocabulario muy general que matchea con muchas cosas.

**Lecturas globales:**

- Los **scores de similaridad son bajos** (máximo ≈ 0.49, muchos en 0.14–0.20). Es esperable: los vectores TF-IDF en este corpus son muy sparsos.
- Los errores no son aleatorios: aparecen casi siempre entre **clases hermanas** del propio dataset (`comp.sys.mac.hardware` ↔ `comp.sys.ibm.pc.hardware`, `comp.os.ms-windows.misc` ↔ `comp.windows.x`, `talk.religion.misc` ↔ `talk.politics.mideast`). Es decir, TF-IDF + coseno **captura tema**, no etiquetas finas.
- La **longitud del documento** es un factor crítico: los dos docs cortos del muestreo (#4965 y #7404) son justamente los de peor desempeño.


## Parte 2 — Clasificador zero-shot por prototipos


Implementamos un clasificador **1-NN sobre similaridad coseno** en el espacio TF-IDF: para cada documento del test buscamos el documento de train más parecido y le asignamos su etiqueta. No hay entrenamiento explícito de un modelo, solo comparaciones — por eso lo llamamos *zero-shot* (cada doc de train hace de prototipo de su clase).

La matriz completa `cosine_similarity(X_test, X_train)` es de (7532, 11314) ≈ 680 MB en float64, así que la calculamos por **chunks** para no estresar la memoria.


In [14]:
def predecir_zero_shot(X_test, X_train, y_train, batch_size=500):
    preds = np.empty(X_test.shape[0], dtype=y_train.dtype)
    for inicio in range(0, X_test.shape[0], batch_size):
        fin = min(inicio + batch_size, X_test.shape[0])
        sims = cosine_similarity(X_test[inicio:fin], X_train)
        vecino = sims.argmax(axis=1)
        preds[inicio:fin] = y_train[vecino]
    return preds

y_pred_zs = predecir_zero_shot(X_test, X_train, y_train, batch_size=500)
f1_zs = f1_score(y_test, y_pred_zs, average='macro')
print(f'F1-macro zero-shot 1-NN coseno: {f1_zs:.4f}')


F1-macro zero-shot 1-NN coseno: 0.5050


In [15]:
# Comparación rápida contra el baseline Multinomial NB (ya calculado más arriba)
f1_baseline = f1_score(y_test, y_pred, average='macro')
print(f'F1-macro MultinomialNB baseline: {f1_baseline:.4f}')
print(f'F1-macro zero-shot 1-NN coseno : {f1_zs:.4f}')
print(f'Diferencia                     : {f1_zs - f1_baseline:+.4f}')


F1-macro MultinomialNB baseline: 0.5854
F1-macro zero-shot 1-NN coseno : 0.5050
Diferencia                     : -0.0804


### Interpretación — Parte 2

**Resultados:**
- F1-macro **Multinomial NB baseline**: `0.5854`
- F1-macro **zero-shot 1-NN coseno**: `0.5050`
- Diferencia: **−0.0804** (≈ 8 puntos por debajo del baseline)

**Por qué pierde:**

- Naïve Bayes integra evidencia de **todas las palabras** del documento para todas las clases; 1-NN se juega todo a un único vecino. Si ese vecino es corto, atípico o queda en un newsgroup hermano, el voto está perdido.
- Como vimos en Parte 1, los scores de similaridad son chicos y muchos vecinos top-1 caen en **clases temáticamente cercanas pero con otra etiqueta**. NB no sufre tanto este problema porque pondera la evidencia con la verosimilitud de todo el vocabulario.
- `remove=('headers','footers','quotes')` deja muchos documentos casi vacíos. Para 1-NN eso es peor que para NB: con dos palabras compartidas alcanza para "ganar" la similaridad top-1, pero esas dos palabras no alcanzan para definir el tema.

**Lo que aporta de todos modos:**

- **No requiere entrenar nada**: solo precomputar la matriz TF-IDF. Es un baseline barato y se extiende a clases nuevas sin re-entrenar (basta agregar prototipos).
- 0.50 de F1-macro sobre 20 clases sigue siendo **10× mejor que random** (~0.05).
- Caminos para mejorarlo manteniendo el espíritu zero-shot:
  - **k-NN** con voto de varios vecinos (más robusto al ruido del top-1).
  - **Centroide por clase** como prototipo (esquema Rocchio): un único vector por clase, pelea más parejo contra NB.


## Parte 3 — Optimización de Naïve Bayes (F1-macro)


Probamos un **grid manual** sobre el vectorizador y el modelo. La restricción de la consigna es no tocar `ngram_range`, así que dejamos `(1,1)` siempre. Variamos:

- **Vectorizador**: `TfidfVectorizer` vs `CountVectorizer`; `min_df`, `max_df`, `stop_words`, `sublinear_tf`, `norm`.
- **Modelo**: `MultinomialNB` vs `ComplementNB`; `alpha` (suavizado de Laplace).

Para cada combinación reportamos F1-macro en test y al final ordenamos.


In [16]:
def evaluar(vectorizador, modelo, train_data, train_y, test_data, test_y):
    Xtr = vectorizador.fit_transform(train_data)
    Xte = vectorizador.transform(test_data)
    modelo.fit(Xtr, train_y)
    pred = modelo.predict(Xte)
    return f1_score(test_y, pred, average='macro'), Xtr.shape[1]


In [17]:
configs = [
    # (descripción, vectorizador, modelo)
    ('TFIDF base + MultinomialNB',
     TfidfVectorizer(), MultinomialNB()),
    ('TFIDF base + ComplementNB',
     TfidfVectorizer(), ComplementNB()),
    ('TFIDF stopwords + MultinomialNB(alpha=0.1)',
     TfidfVectorizer(stop_words='english'), MultinomialNB(alpha=0.1)),
    ('TFIDF stopwords + ComplementNB(alpha=0.1)',
     TfidfVectorizer(stop_words='english'), ComplementNB(alpha=0.1)),
    ('TFIDF sw+min_df=2+max_df=0.95 + ComplementNB(alpha=0.1)',
     TfidfVectorizer(stop_words='english', min_df=2, max_df=0.95),
     ComplementNB(alpha=0.1)),
    ('TFIDF sw+min_df=5+sublinear + ComplementNB(alpha=0.3)',
     TfidfVectorizer(stop_words='english', min_df=5, sublinear_tf=True),
     ComplementNB(alpha=0.3)),
    ('TFIDF sw+min_df=2+sublinear + ComplementNB(alpha=0.05)',
     TfidfVectorizer(stop_words='english', min_df=2, sublinear_tf=True),
     ComplementNB(alpha=0.05)),
    ('TFIDF sw+min_df=2 + ComplementNB(alpha=0.5)',
     TfidfVectorizer(stop_words='english', min_df=2),
     ComplementNB(alpha=0.5)),
    ('Count sw+min_df=2 + MultinomialNB(alpha=0.1)',
     CountVectorizer(stop_words='english', min_df=2),
     MultinomialNB(alpha=0.1)),
    ('Count sw+min_df=2 + ComplementNB(alpha=0.1)',
     CountVectorizer(stop_words='english', min_df=2),
     ComplementNB(alpha=0.1)),
]

resultados = []
for desc, vect, model in configs:
    f1, vocab = evaluar(vect, model,
                        newsgroups_train.data, y_train,
                        newsgroups_test.data, y_test)
    resultados.append((f1, vocab, desc))
    print(f'{f1:.4f}  | vocab={vocab:6d} | {desc}')


0.5854  | vocab=101631 | TFIDF base + MultinomialNB


0.6930  | vocab=101631 | TFIDF base + ComplementNB


0.6726  | vocab=101322 | TFIDF stopwords + MultinomialNB(alpha=0.1)


0.6919  | vocab=101322 | TFIDF stopwords + ComplementNB(alpha=0.1)


0.6887  | vocab= 39115 | TFIDF sw+min_df=2+max_df=0.95 + ComplementNB(alpha=0.1)


0.6782  | vocab= 17797 | TFIDF sw+min_df=5+sublinear + ComplementNB(alpha=0.3)


0.6836  | vocab= 39115 | TFIDF sw+min_df=2+sublinear + ComplementNB(alpha=0.05)


0.6974  | vocab= 39115 | TFIDF sw+min_df=2 + ComplementNB(alpha=0.5)


0.6284  | vocab= 39115 | Count sw+min_df=2 + MultinomialNB(alpha=0.1)


0.6387  | vocab= 39115 | Count sw+min_df=2 + ComplementNB(alpha=0.1)


In [18]:
# Orden de mejor a peor
print('=' * 80)
print('Ranking F1-macro')
print('=' * 80)
for f1, vocab, desc in sorted(resultados, key=lambda x: -x[0]):
    print(f'{f1:.4f}  | vocab={vocab:6d} | {desc}')


Ranking F1-macro
0.6974  | vocab= 39115 | TFIDF sw+min_df=2 + ComplementNB(alpha=0.5)
0.6930  | vocab=101631 | TFIDF base + ComplementNB
0.6919  | vocab=101322 | TFIDF stopwords + ComplementNB(alpha=0.1)
0.6887  | vocab= 39115 | TFIDF sw+min_df=2+max_df=0.95 + ComplementNB(alpha=0.1)
0.6836  | vocab= 39115 | TFIDF sw+min_df=2+sublinear + ComplementNB(alpha=0.05)
0.6782  | vocab= 17797 | TFIDF sw+min_df=5+sublinear + ComplementNB(alpha=0.3)
0.6726  | vocab=101322 | TFIDF stopwords + MultinomialNB(alpha=0.1)
0.6387  | vocab= 39115 | Count sw+min_df=2 + ComplementNB(alpha=0.1)
0.6284  | vocab= 39115 | Count sw+min_df=2 + MultinomialNB(alpha=0.1)
0.5854  | vocab=101631 | TFIDF base + MultinomialNB


### Interpretación — Parte 3

**Ranking obtenido** (F1-macro en test):

| # | F1     | Vocab   | Config |
|---|--------|---------|--------|
| 1 | **0.6974** |  39 115 | TFIDF + stopwords + min_df=2 + **ComplementNB(α=0.5)** |
| 2 | 0.6930 | 101 631 | TFIDF base + ComplementNB(α=1.0 default) |
| 3 | 0.6919 | 101 322 | TFIDF + stopwords + ComplementNB(α=0.1) |
| 4 | 0.6887 |  39 115 | TFIDF + stopwords + min_df=2 + max_df=0.95 + ComplementNB(α=0.1) |
| 5 | 0.6836 |  39 115 | TFIDF + stopwords + min_df=2 + sublinear_tf + ComplementNB(α=0.05) |
| 6 | 0.6782 |  17 797 | TFIDF + stopwords + min_df=5 + sublinear_tf + ComplementNB(α=0.3) |
| 7 | 0.6726 | 101 322 | TFIDF + stopwords + MultinomialNB(α=0.1) |
| 8 | 0.6387 |  39 115 | Count + stopwords + min_df=2 + ComplementNB(α=0.1) |
| 9 | 0.6284 |  39 115 | Count + stopwords + min_df=2 + MultinomialNB(α=0.1) |
| 10 | 0.5854 | 101 631 | TFIDF base + MultinomialNB (**baseline**) |

**Mejor configuración: 0.6974 vs baseline 0.5854 → +11.2 puntos de F1-macro.**

**Patrones que salen de la tabla:**

- **ComplementNB domina**: las 7 mejores configuraciones usan ComplementNB. Incluso con hiperparámetros default (config 2: TFIDF base + ComplementNB) ya superamos por +10 puntos al MultinomialNB baseline. La razón es bien conocida: 20NG tiene clases de tamaños distintos (`alt.atheism` tiene ~480 docs, `rec.sport.hockey` tiene ~600), y ComplementNB estima los parámetros con el **complemento** de cada clase, evitando que las clases grandes aplasten a las chicas.
- **TF-IDF > Count**: las dos configuraciones con `CountVectorizer` quedan en el fondo (#8 y #9). La normalización L2 + IDF que aporta TF-IDF claramente ayuda al modelo lineal subyacente.
- **`stop_words='english'`** sube de forma consistente: las palabras funcionales no aportan a la verosimilitud por clase y solo ensucian.
- **`min_df=2`** reduce vocabulario de ~101 k a ~39 k (saca palabras hapax) sin perder casi nada de performance — y la mejor config justamente la usa.
- **`min_df=5`** ya es demasiado agresivo: el vocab cae a ~17 k y bajamos a 0.6782.
- **`alpha`**: la sintonía no es monótona. La mejor config tiene **α = 0.5**, no α = 0.1 ni α = 0.05. Las diferencias entre α ∈ {0.05, 0.1, 0.5, 1.0} son chicas (< 1 punto), pero el α default = 1.0 con TFIDF + sin stopwords igual queda decente (config 2). Un grid de α más fino refinaría esto.
- **`sublinear_tf=True`** no aportó en esta corrida: las dos configs que lo usan (#5 y #6) quedaron debajo del ganador, que no lo usa.

**Conclusión práctica:** con cambios chicos al pipeline (TFIDF + stopwords + min_df=2 + ComplementNB con α moderado) se obtiene una mejora muy significativa. El mayor impulso vino de **cambiar el modelo (Complement vs Multinomial)**, no de tunear el vectorizador.


## Parte 4 — Similaridad entre palabras (matriz término-documento)


Si transponemos la matriz documento-término obtenemos una matriz término-documento: ahora cada **fila es una palabra** representada por su perfil de aparición a lo largo de los ~11 k documentos del train. Dos palabras son similares si tienden a aparecer en los mismos documentos (es la idea de **distributional semantics**: "a word is known by the company it keeps").

Elegimos 5 palabras **manualmente** apuntando a vocabulario temático bien definido dentro de 20NG, para que los vecinos sean interpretables:

- `god` → religión
- `car` → autos
- `windows` → sistemas operativos
- `space` → exploración espacial
- `hockey` → deportes


In [19]:
X_palabras = X_train.T.tocsr()  # término-documento, sparse-CSR para indexar filas
print(f'Matriz término-documento: {X_palabras.shape}  (palabras × documentos)')


Matriz término-documento: (101631, 11314)  (palabras × documentos)


In [20]:
palabras_objetivo = ['god', 'car', 'windows', 'space', 'hockey']

def top_palabras_similares(palabra, vocab, idx2word, X_palabras, n=5):
    if palabra not in vocab:
        print(f'   (la palabra "{palabra}" no está en el vocabulario)')
        return
    idx = vocab[palabra]
    sims = cosine_similarity(X_palabras[idx], X_palabras)[0]
    orden = np.argsort(sims)[::-1]
    vecinos = orden[orden != idx][:n]
    return [(idx2word[i], sims[i]) for i in vecinos]

for w in palabras_objetivo:
    print(f'\nMás similares a "{w}":')
    resultado = top_palabras_similares(w, tfidfvect.vocabulary_, idx2word, X_palabras, n=5)
    if resultado:
        for vecina, s in resultado:
            print(f'   {vecina:20s}  sim={s:.3f}')



Más similares a "god":
   jesus                 sim=0.269
   bible                 sim=0.262
   that                  sim=0.256
   existence             sim=0.255
   christ                sim=0.251

Más similares a "car":
   cars                  sim=0.180
   criterium             sim=0.177
   civic                 sim=0.175
   owner                 sim=0.169
   dealer                sim=0.168

Más similares a "windows":
   dos                   sim=0.304
   ms                    sim=0.232
   microsoft             sim=0.222
   nt                    sim=0.214
   for                   sim=0.193

Más similares a "space":
   nasa                  sim=0.330
   seds                  sim=0.297
   shuttle               sim=0.293
   enfant                sim=0.280
   seti                  sim=0.246

Más similares a "hockey":
   ncaa                  sim=0.274
   nhl                   sim=0.265
   affiliates            sim=0.248
   xenophobes            sim=0.243
   sportschannel         sim=0.

### Interpretación — Parte 4

Los vecinos obtenidos:

| Palabra | Top-5 vecinos |
|---------|---------------|
| `god`     | `jesus` (0.27), `bible` (0.26), `that` (0.26), `existence` (0.26), `christ` (0.25) |
| `car`     | `cars` (0.18), `criterium` (0.18), `civic` (0.18), `owner` (0.17), `dealer` (0.17) |
| `windows` | `dos` (0.30), `ms` (0.23), `microsoft` (0.22), `nt` (0.21), `for` (0.19) |
| `space`   | `nasa` (0.33), `seds` (0.30), `shuttle` (0.29), `enfant` (0.28), `seti` (0.25) |
| `hockey`  | `ncaa` (0.27), `nhl` (0.27), `affiliates` (0.25), `xenophobes` (0.24), `sportschannel` (0.22) |

**Lo que funciona bien:**

- **Validación de la hipótesis distribucional**: sin entrenar ningún embedding, la pura matriz término-documento ya captura asociaciones semánticas razonables. Para `god` aparecen `jesus`, `bible`, `christ`, `existence`. Para `space`, `nasa`/`shuttle`/`seti`/`seds` (los últimos dos son organizaciones espaciales reales, *SEDS = Students for the Exploration and Development of Space*). Para `windows`, todo el ecosistema MS (`dos`, `ms`, `microsoft`, `nt`).
- La similaridad es **temática, no semántica fina**: `car` no queda al lado de su sinónimo `automobile` sino del *contexto* en el que aparece (`engine`, `dealer`, `owner`, modelos como `civic`).
- **Morfología**: `car` y `cars` aparecen como vecinos cercanos — sin lematización, las formas plurales/conjugadas son palabras distintas. Un stemmer las unificaría.

**Ruido observable:**

- Aparecen palabras funcionales como vecinas: `that` cerca de `god`, `for` cerca de `windows`. Son stopwords que coocurren con todo y no aportan significado. Una limpieza con `stop_words='english'` antes de transponer ayudaría.
- Hay términos **espurios por coocurrencia casual**: `enfant` cerca de `space`, `xenophobes` cerca de `hockey`. Probablemente vienen de uno o dos documentos donde aparecieron juntos por azar. Es el clásico problema de la similaridad sobre vectores muy sparsos: pocos co-eventos pesan mucho.
- Las similaridades absolutas son **bajas** (max ≈ 0.33). Los vectores de palabras son mucho más sparsos que los de documentos.

**Conclusión:** la matriz término-documento sirve como prueba de concepto de que la distribución del contexto codifica significado, pero para extraer asociaciones de calidad conviene **reducir dimensionalidad** (LSA / SVD truncado sobre esta matriz, o directamente Word2Vec/GloVe). Ese fue, históricamente, el camino que llevó a los embeddings densos modernos.
